# Libraries

In [89]:
!pip install gensim plotly scikit-learn


In [ ]:
import os
import re
import multiprocessing
from collections import Counter
import pandas as pd
import numpy as np
import ast
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import plotly.express as px

import plotly.io as pio

## Environment setup

In your Google Drive, create a folder "CLT" and upload the csv with its original name (downloaded from Kaggle).

Switch: "local" (VS Code + GitHub) | "colab" (Google Colab + Drive)

In [91]:
ENV = "colab"
pio.renderers.default = "colab" if ENV == "colab" else "notebook"

In [92]:
if ENV == "colab":
    # from google.colab import userdata
    from google.colab import drive
    drive.mount('/content/drive')
    # Load Gemini API key from Colab Secrets (set via the key icon in the sidebar)
    # GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Stage 2 Dataset Preparation and Export


## 1.1 Introduction

At this point Stage 1 preprocessing pipeline is complete.
Dataset after Stage 1 is about 1.5GB and we decided to
save only necessary data to `ai_media_norm.csv` containing only the 8 columns needed for Stage 2, discarding all intermediate token and lemma variants that were used internally during cleaning.
This also avoids repeating the computationally expensive and long spaCy processing (tokenization, lemmatization).

**Saved columns:** `date`, `domain`, `url`, `tags`, `title`, `content`, `title_lemma_norm`, `content_lemma_norm`.

This keeps the file small and focused — Stage 2 only needs the original metadata, the cleaned readable text, and the final normalized lemma lists for embedding training.

Following commented code should show what was done at the end of the preprocessing step in Stage 1.



In [ ]:
# This code doesnt need to be run, its just a demonstration of how we saved the normalized data
# in Stage 1 file (clt.ipynb).

# Save only the columns needed for Stage 2 as a checkpoint
# stage2_input_cols = [
#     'date', 'domain', 'url', 'tags',
#     'title', 'content',
#     'title_lemma_norm', 'content_lemma_norm',
# ]

# if ENV == "colab":
#     norm_path = '/content/drive/My Drive/CLT/ai_media_norm.csv'
# else:
#     norm_path = 'data/ai_media_norm.csv'

# os.makedirs(os.path.dirname(norm_path), exist_ok=True)
# df_norm[stage2_input_cols].to_csv(norm_path, index=False, encoding='utf-8')

# print("Saved to:", norm_path)
# print("Shape:", df_norm[stage2_input_cols].shape)

## 1.2 Load Stage 1 Checkpoint

We load the preprocessed dataset (`ai_media_norm.csv`)

Since CSV does not preserve Python list types, we use `ast.literal_eval` to restore `title_lemma_norm`, `content_lemma_norm`, and `tags` back to proper Python lists. The `date` column is also re-parsed to `datetime`.


In [50]:
# Load the Stage 1 checkpoint
if ENV == "colab":
    norm_path = '/content/drive/My Drive/CLT/ai_media_norm.csv'
else:
    norm_path = 'data/ai_media_norm.csv'

df_norm_loaded = pd.read_csv(norm_path, encoding='utf-8')

# Restore list columns serialized as strings by CSV
for col in ['title_lemma_norm', 'content_lemma_norm', 'tags']:
    df_norm_loaded[col] = df_norm_loaded[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

df_norm_loaded['date'] = pd.to_datetime(df_norm_loaded['date'])

print("Loaded shape:", df_norm_loaded.shape)


Loaded shape: (16518, 8)


## 1.3 Build Stage 2 Columns

- **`doc_id`**: Stable zero-padded identifier (e.g. `doc_00042`) — survives CSV export and links model outputs back to source articles.
- **`text_clean_light`**: Readable text from cleaned `title` + `content` — used for display, search, and labeling. Stop words kept, natural sentence structure preserved.
- **`text_for_embedding`**: Input for Word2Vec / FastText / Doc2Vec — `title_lemma_norm` + `content_lemma_norm` joined as a space-separated string. Stop words removed, noise filtered.
- **`token_count`**: Number of tokens in `text_for_embedding`, used for quality filtering in the next step.
- **`split`**: Reproducible 80/20 train/val assignment with fixed random seed (42).


In [51]:
df_stage2 = df_norm_loaded.copy().reset_index(drop=True)

# Unique document ID
df_stage2['doc_id'] = ['doc_' + str(i).zfill(5) for i in range(len(df_stage2))]

# Readable combined text: cleaned title + cleaned content
df_stage2['text_clean_light'] = (
    df_stage2['title'].fillna('') + '. ' + df_stage2['content'].fillna('')
).str.strip('. ')

# Embedding text: final normalized lemma tokens from title + content joined
df_stage2['text_for_embedding'] = df_stage2.apply(
    lambda row: ' '.join(
        (row['title_lemma_norm'] if isinstance(row['title_lemma_norm'], list) else []) +
        (row['content_lemma_norm'] if isinstance(row['content_lemma_norm'], list) else [])
    ),
    axis=1
)

# Token count
df_stage2['token_count'] = df_stage2['text_for_embedding'].str.split().str.len()

# Reproducible 80/20 train/val split
np.random.seed(42)
split_mask = np.random.rand(len(df_stage2)) < 0.8
df_stage2['split'] = np.where(split_mask, 'train', 'val')


## 1.4 Filter, Export and Verify

Before exporting we apply three quality filters to ensure only meaningful documents reach the embedding model:

1. **Remove empty embedding texts** — rows where `text_for_embedding` is blank or whitespace-only are dropped. These would produce zero-length inputs that crash or corrupt model training.
2. **Remove very short texts** — rows with fewer than 5 tokens are dropped. Documents this short carry almost no semantic signal and can distort word co-occurrence statistics in Word2Vec / FastText.
3. **Drop embedding duplicates** — rows whose `text_for_embedding` is identical to another row are deduplicated. Repeated documents would over-weight certain terms in the trained embeddings.

The index is then reset to close any gaps left by the filters, ensuring clean positional access. The final dataframe is narrowed to the 11 export columns and written to three CSV files — `ai_media_stage2_full.csv`, `ai_media_stage2_train.csv`, and `ai_media_stage2_val.csv` (all UTF-8). A summary is printed showing the final shape, train/val counts, and a sample of the key columns.


In [52]:
# Remove empty / very short embedding texts
df_stage2 = df_stage2[df_stage2['text_for_embedding'].str.strip() != '']
df_stage2 = df_stage2[df_stage2['token_count'] >= 5]

# Drop duplicates based on embedding text
df_stage2 = df_stage2.drop_duplicates(subset='text_for_embedding').reset_index(drop=True)

# Final column selection
export_cols = ['doc_id', 'date', 'domain', 'url', 'tags',
               'title', 'content', 'text_clean_light',
               'text_for_embedding', 'token_count', 'split']
df_export = df_stage2[export_cols]

if ENV == "colab":
    base_path = '/content/drive/My Drive/CLT/stage_2/'
else:
    base_path = 'data/stage_2/'

os.makedirs(base_path, exist_ok=True)

# Full dataset
df_export.to_csv(base_path + 'ai_media_stage2_full.csv', index=False, encoding='utf-8')

# Train / val splits — ready for downstream supervised tasks
df_export[df_export['split'] == 'train'].to_csv(base_path + 'ai_media_stage2_train.csv', index=False, encoding='utf-8')
df_export[df_export['split'] == 'val'].to_csv(base_path + 'ai_media_stage2_val.csv', index=False, encoding='utf-8')

# Summary
print("Export shape:", df_export.shape)
print("\nSplit counts:")
print(df_export['split'].value_counts())
print("\nSample (first 3 rows):")
sample = df_export[['doc_id', 'text_clean_light', 'text_for_embedding', 'token_count', 'split']].head(3).copy()
sample['text_clean_light'] = sample['text_clean_light'].str[:80] + '...'
sample['text_for_embedding'] = sample['text_for_embedding'].str[:80] + '...'
print(sample.to_string())

Export shape: (16518, 11)

Split counts:
split
train    13224
val       3294
Name: count, dtype: int64

Sample (first 3 rows):
      doc_id                                                                     text_clean_light                                                                   text_for_embedding  token_count  split
0  doc_00000  ricoh to provide customer support for agility robotics digit humanoid. the digit...  ricoh provide customer support agility robotic digit humanoid digit humanoid wor...          501  train
1  doc_00001  ai design trends to watch in 2025 from photorealism to personalization. artifici...  ai design trend watch photorealism personalization artificial intelligence ai ra...          804    val
2  doc_00002  ais generate more novel and exciting research ideas than human experts. the firs...  ais generate novel exciting research idea human expert statistically significant...          533  train


# 2. Word2Vec Embeddings


We train a **Skip-gram Word2Vec** model on the `text_for_embedding` column of the training split.
The input tokens are the pre-processed lemma sequences produced in Stage 1 (stop words removed, normalised).

**Role in the broader pipeline:**  
Word2Vec operates on individual lemma tokens, not BPE subwords, so its vectors do not plug directly into a TinyLlama-style transformer.
They serve as:
- a **domain vocabulary sanity check** — nearest-neighbour probes reveal whether the corpus signal is coherent;
- **embedding initialisation** for lightweight classification or retrieval heads built on top of the LLM;
- **input features** for non-neural baselines (TF-IDF + W2V centroid classifiers, etc.).

**Hyperparameters chosen:**

| Parameter | Value | Rationale |
|---|---|---|
| `vector_size` | 200 | Good balance for ~16 k documents |
| `window` | 5 | Standard context window |
| `min_count` | 5 | Drop tokens appearing fewer than 5 times |
| `sg` | 1 | Skip-gram; better for rare domain terms |
| `epochs` | 10 | Sufficient for convergence on this corpus size |


## 2.1 Load & Tokenize

We load the training split from `ai_media_stage2_train.csv`.
Each row's `text_for_embedding` is already a space-separated string of lemma tokens.

A small normalization pass is applied before training to remap known lemmatization artifacts:

| Original token | Replacement | Reason |
|---|---|---|
| `datum` | `data` | spaCy lemmatises "data" → "datum" |
| `medium` | `media` | spaCy lemmatises "media" → "medium" |
| `u200b` | *(dropped)* | Unicode zero-width space (U+200B) leaked as literal token |
| `https`, `http` | *(dropped)* | URL protocol fragments from incompletely stripped hyperlinks |
| hex tokens | *(dropped)* | UTF-8 mojibake artifacts (`x9d`, `x80`, `x9cthe`, etc.) |

Counts before and after are printed so any additional odd tokens can be caught and added to `TOKEN_MAP`.


In [59]:
if ENV == "colab":
    train_path = '/content/drive/My Drive/CLT/stage_2/ai_media_stage2_train.csv'
else:
    train_path = 'data/stage_2/ai_media_stage2_train.csv'

df_train = pd.read_csv(train_path, encoding='utf-8')

# Split space-separated token strings into lists — required format for gensim
sentences = [text.split() for text in df_train['text_for_embedding'].dropna()]

# --- Token normalization ---
# TOKEN_MAP: explicit remappings (old → new, empty string = drop).
# HEX_PATTERN: drops any token starting with a hex escape sequence (x9d, x9cthe, etc.)
TOKEN_MAP = {
    'datum' : 'data',    # spaCy lemmatises "data" → "datum"
    'medium': 'media',   # spaCy lemmatises "media" → "medium"
    'u200b' : '',        # Unicode zero-width space leaked as literal token — drop
    'https' : '',        # URL protocol fragment — drop
    'http'  : '',        # URL protocol fragment — drop
}
HEX_PATTERN = re.compile(r'^x[0-9a-f]{2}')   # matches x9d, x80, x9cthe, etc.

def _keep(tok: str) -> str:
    """Return the normalised token, or '' to signal drop."""
    tok = TOKEN_MAP.get(tok, tok)          # apply explicit remapping first
    if HEX_PATTERN.match(tok):             # then drop hex artifacts
        return ''
    return tok

# Check counts before fix
from collections import Counter
flat_before   = [t for s in sentences for t in s]
flat_before   = [t for s in sentences for t in s]
counts_before = Counter(flat_before)
hex_before    = {t: c for t, c in counts_before.items() if HEX_PATTERN.match(t)}
print("=== Token counts BEFORE normalization ===")
for tok in TOKEN_MAP:
    print(f"  {tok:<12} : {counts_before.get(tok, 0):,}")
print(f"  hex artifacts: {sum(hex_before.values()):,} occurrences across {len(hex_before):,} unique tokens")

# Apply normalization
sentences = [
    [out for t in s for out in [_keep(t)] if out]
    for s in sentences
]

# Check counts after fix
flat_after   = [t for s in sentences for t in s]
counts_after = Counter(flat_after)
hex_after    = {t: c for t, c in counts_after.items() if HEX_PATTERN.match(t)}
print("\n=== Token counts AFTER normalization ===")
for orig, repl in TOKEN_MAP.items():
    print(f"  {orig:<12} → {repl:<12} : {counts_after.get(repl, 0):,}")
print(f"  hex artifacts remaining: {sum(hex_after.values()):,}")

print(f"\nDocuments  : {len(sentences):,}")
print(f"Total tokens: {sum(len(s) for s in sentences):,}")
print(f"Sample     : {sentences[0][:12]} ...")

=== Token counts BEFORE normalization ===
  datum        : 74,577
  medium       : 10,143
  u200b        : 630
  https        : 3,023
  http         : 351
  hex artifacts: 40,403 occurrences across 2,461 unique tokens

=== Token counts AFTER normalization ===
  datum        → data         : 91,666
  medium       → media        : 10,793
  u200b        →              : 0
  https        →              : 0
  http         →              : 0
  hex artifacts remaining: 0

Documents  : 13,224
Total tokens: 11,296,257
Sample     : ['ricoh', 'provide', 'customer', 'support', 'agility', 'robotic', 'digit', 'humanoid', 'digit', 'humanoid', 'work', 'distribution'] ...


In [62]:
flat_tokens = Counter(t for s in sentences for t in s)
TOP_FREQ    = 50   # show this many of the most frequent suspicious tokens per category

suspicious = {
    'hex_artifacts'  : {},   # x80, xc2, etc.
    'non_ascii'      : {},   # contains chars outside basic ASCII
    'numeric_only'   : {},   # pure numbers like "2023", "100"
    'single_char'    : {},   # single character tokens
    'punctuation'    : {},   # contains non-alphanumeric characters
}

for tok, cnt in flat_tokens.items():
    if re.fullmatch(r'x[0-9a-f]{2}', tok):
        suspicious['hex_artifacts'][tok] = cnt
    elif not tok.isascii():
        suspicious['non_ascii'][tok] = cnt
    elif tok.isnumeric():
        suspicious['numeric_only'][tok] = cnt
    elif len(tok) == 1:
        suspicious['single_char'][tok] = cnt
    elif not tok.isalpha():
        suspicious['punctuation'][tok] = cnt

for category, tokens in suspicious.items():
    top = sorted(tokens.items(), key=lambda x: x[1], reverse=True)[:TOP_FREQ]
    if top:
        print(f"\n{'='*50}")
        print(f"  {category.upper()}  ({len(tokens):,} unique tokens)")
        print(f"{'='*50}")
        for tok, cnt in top:
            print(f"  {tok:<20} : {cnt:,}")
    else:
        print(f"\n  {category}: ✓ none found")



  hex_artifacts: ✓ none found

  non_ascii: ✓ none found

  numeric_only: ✓ none found

  single_char: ✓ none found

  PUNCTUATION  (8,907 unique tokens)
  3d                   : 1,896
  b2b                  : 1,122
  r1                   : 1,107
  4o                   : 748
  q1                   : 646
  o1                   : 635
  a16z                 : 572
  4th                  : 553
  q4                   : 539
  15th                 : 482
  3rd                  : 457
  q2                   : 452
  co2                  : 440
  q3                   : 425
  o3                   : 415
  sc24                 : 378
  j916                 : 369
  yolov10              : 367
  h100                 : 352
  20th                 : 349
  21st                 : 322
  v3                   : 316
  1b                   : 309
  30th                 : 292
  g7                   : 283
  1st                  : 276
  4k                   : 262
  a2a                  : 260
  eur1                 : 25

## 2.1.1 Suspicious Token Scan


**Expected tokens in the `punctuation` category** — alphanumeric tokens containing digits are not artifacts; they are legitimate corpus vocabulary:

| Token(s) | Type | Examples |
|---|---|---|
| `3d`, `2d`, `4k` | Dimensional / display spec | 3D rendering, 4K video |
| `b2b`, `b2c` | Business model abbreviation | business-to-business, business-to-consumer |
| `r1`, `o1`, `o3`, `4o` | AI model names | DeepSeek R1, OpenAI o1/o3, GPT-4o |
| `q1` – `q4` | Fiscal quarters | Q1 earnings, Q4 results |
| `8b`, `70b`, `1b`, `3b` | Model parameter scale | 8-billion-parameter model |
| `v1`, `v2`, `v3` | Version numbers | model version, API version |
| `h100`, `gb200` | GPU / hardware names | NVIDIA H100, GB200 |
| `10x`, `5x` | Growth multipliers | 10× revenue growth |
| `a16z` | VC fund shorthand | Andreessen Horowitz |
| `web3`, `co2`, `g7` | Domain terms | Web3 ecosystem, CO₂ emissions, G7 summit |
| ordinals (`1st`…`30th`) | Date ordinals | published on the 15th |

These should all be **kept** — they carry real semantic signal for the AI-media domain.


## 2.2 Train Word2Vec

Training uses all available CPU cores.


In [63]:
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=200,   # embedding dimensions
    window=5,          # context window
    min_count=5,       # ignore tokens appearing fewer than 5 times
    sg=1,              # 1 = Skip-gram, 0 = CBOW
    workers=multiprocessing.cpu_count(),
    epochs=10,
    seed=42,
)

print(f"Vocabulary size : {len(w2v_model.wv):,} tokens")
print(f"Vector dimensions: {w2v_model.wv.vector_size}")


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Vocabulary size : 42,155 tokens
Vector dimensions: 200


## 2.3 Save Model

Two artefacts are saved to `data/stage_2/embeddings/`:

- **`w2v_ai_media.model`** — full gensim model (can be reloaded for further training or inference).
- **`w2v_ai_media_vectors.txt`** — plain-text word vectors in `word2vec` format, compatible with external tools (FastText, Magnitude, `load_facebook_vectors`, etc.).


In [64]:
if ENV == "colab":
    model_dir = '/content/drive/My Drive/CLT/stage_2/embeddings/'
else:
    model_dir = 'data/stage_2/embeddings/'

os.makedirs(model_dir, exist_ok=True)

# Full gensim model (reloadable)
w2v_model.save(model_dir + 'w2v_ai_media.model')

# Plain-text vectors (word2vec format — compatible with external tools)
w2v_model.wv.save_word2vec_format(model_dir + 'w2v_ai_media_vectors.txt', binary=False)

print("Saved to:", model_dir)


Saved to: data/stage_2/embeddings/


## 2.4 Evaluate

Nearest-neighbour probes confirm the model has learned domain-coherent representations.
We test a small set of AI-media terms — adjust `probe_words` to match your corpus vocabulary.
We also print the top-20 tokens by frequency to quickly spot any noise that survived the earlier cleaning steps.


In [65]:
wv = w2v_model.wv

# --- Nearest-neighbour probes ---
probe_words = ['ai', 'model', 'work', 'bias', 'security']

print("=== Nearest-neighbour probes (top 5) ===")
for word in probe_words:
    if word in wv:
        neighbours = wv.most_similar(word, topn=5)
        neighbours_str = ', '.join(f"{w} ({s:.2f})" for w, s in neighbours)
        print(f"  {word:<12} → {neighbours_str}")
    else:
        print(f"  {word:<12} → not in vocabulary")

# --- Top-20 tokens by frequency ---
print("\n=== Top-20 tokens by frequency ===")
vocab_counts = sorted(wv.key_to_index.keys(),
                      key=lambda w: w2v_model.wv.get_vecattr(w, 'count'),
                      reverse=True)[:20]
for i, w in enumerate(vocab_counts, 1):
    count = w2v_model.wv.get_vecattr(w, 'count')
    print(f"  {i:>2}. {w:<20} {count:,}")


=== Nearest-neighbour probes (top 5) ===
  ai           → genai (0.65), generative (0.59), artificial (0.56), assistanta (0.55), digitate (0.55)
  model        → autorater (0.69), modelsa (0.67), grm (0.66), mllm (0.66), dllm (0.65)
  work         → talente (0.59), teramind (0.56), collaborate (0.56), costigan (0.54), edplus (0.52)
  bias         → perpetuate (0.67), biased (0.67), prejudice (0.65), fairness (0.62), discrimination (0.60)
  security     → cybersecurity (0.69), posture (0.56), recoverability (0.55), blackpoint (0.55), carbonite (0.55)

=== Top-20 tokens by frequency ===
   1. ai                   163,854
   2. data                 91,666
   3. use                  78,932
   4. model                72,459
   5. company              65,933
   6. new                  47,417
   7. time                 45,220
   8. like                 44,973
   9. technology           43,465
  10. system               41,884
  11. year                 39,739
  12. business             37,399

**Embedding evaluation**

The Word2Vec model captures meaningful semantic relationships within the AI discourse dataset. For example, the word “bias” is strongly associated with fairness, discrimination, and prejudice, reflecting discussions around AI ethics. Similarly, “ai” clusters with terms such as generative and genai, indicating the prominence of generative AI topics in the corpus.

**Dataset structure**

The most frequent tokens show that the dataset primarily discusses AI technologies and industry adoption. Terms such as model, data, technology, and company dominate the vocabulary, confirming the dataset’s focus on technological developments and business applications of AI

# 3. Cluster Visualisation

We project the Word2Vec **vocabulary** into 2D using **t-SNE**, then assign each word to a thematic cluster using **K-Means**.
Each point in the interactive plot is one vocabulary token. Points that are close together share similar contexts in the corpus — hovering reveals the word and its frequency.

**Pipeline:**
1. Take the top `TOP_N` most-frequent words (keeps the plot readable and focuses on signal-rich terms).
2. L2-normalise the vectors (equivalent to cosine similarity for K-Means).
3. Reduce to 2D with t-SNE (`perplexity=40`, `n_iter=1000`).
4. Cluster with K-Means (`N_CLUSTERS` groups).
5. Render as an interactive Plotly scatter — point size scales with token frequency.


## 3.1 Configure & Extract Vectors

Adjust `TOP_N` and `N_CLUSTERS` to trade off detail vs. readability.
A good starting point is 2 000–4 000 words and 6–10 clusters for an AI-media corpus of this size.


In [ ]:
TOP_N      = 3000   # most-frequent words to include in the plot
N_CLUSTERS = 12     # number of thematic clusters

# Anchor words for each theme — the cluster whose vocabulary best matches
# a set of anchors gets that name automatically (robust to ID shuffling across runs).
THEME_ANCHORS = {
    'AI Models & Data'      : ['ai', 'model', 'data', 'language', 'llm', 'train', 'neural', 'transformer', 'vector', 'embed'],
    'Industry & Business'   : ['company', 'business', 'market', 'customer', 'product', 'revenue', 'startup', 'enterprise', 'invest', 'technology'],
    'Policy & Governance'   : ['policy', 'regulation', 'governance', 'government', 'law', 'right', 'global', 'rule', 'compliance', 'share', 'value'],
    'Research & Society'    : ['research', 'study', 'human', 'social', 'result', 'find', 'scientist', 'university', 'paper', 'new', 'time', 'year'],
    'Systems & Tools'       : ['system', 'software', 'platform', 'deploy', 'infrastructure', 'automate', 'integrate', 'workflow', 'use', 'help'],
    'Media & Information'   : ['content', 'information', 'access', 'news', 'media', 'publish', 'journalist', 'article', 'report'],
    'General Narrative'     : ['say', 'call', 'note', 'describe', 'suggest', 'believe', 'accord', 'add', 'explain', 'like', 'need', 'lead'],
    'Ethics & Safety'       : ['bias', 'safety', 'harm', 'risk', 'ethical', 'responsible', 'privacy', 'fairness', 'trust', 'accountability'],
    'Investment & Finance'  : ['fund', 'invest', 'capital', 'million', 'billion', 'valuation', 'raise', 'acquisition', 'deal', 'vc'],
    'Innovation & Development': ['build', 'develop', 'drive', 'change', 'improve', 'design', 'create', 'launch', 'engineer', 'solution'],
    'Education & Workforce' : ['student', 'education', 'skill', 'job', 'worker', 'learn', 'school', 'career', 'employment', 'talent'],
    'Healthcare & Science'  : ['health', 'medical', 'drug', 'patient', 'clinical', 'diagnosis', 'hospital', 'biology', 'disease', 'science'],
}

# Sort vocabulary by frequency descending, take top N
words_sorted = sorted(
    wv.key_to_index.keys(),
    key=lambda w: wv.get_vecattr(w, 'count'),
    reverse=True
)[:TOP_N]

vectors = np.array([wv[w] for w in words_sorted])
freqs   = np.array([wv.get_vecattr(w, 'count') for w in words_sorted])



print(f"Words selected : {len(words_sorted):,}")
print(f"Vector matrix  : {vectors.shape}")
print(f"Frequency range: {freqs.min():,} – {freqs.max():,}")

Words selected : 3,000
Vector matrix  : (3000, 200)
Frequency range: 498 – 163,854


## 3.2 Optimal K — Elbow & Silhouette

Before fixing `N_CLUSTERS` we sweep K = 2 … 12 on the L2-normalised vectors and plot two diagnostics:

- **Elbow (inertia)** — within-cluster sum of squares; look for the point where the curve bends and further gains shrink.
- **Silhouette score** — mean similarity of each word to its own cluster vs. the next-best cluster; ranges 0–1, higher is better; pick the peak.

Both charts use the same high-dimensional normalised vectors (not the 2D t-SNE projection) so the K choice is grounded in the actual embedding geometry.


In [67]:
from sklearn.metrics import silhouette_score

K_RANGE = range(2, 13)

vectors_norm_k = normalize(vectors)   # L2-normalise for cosine-like K-Means

inertias    = []
silhouettes = []

for k in K_RANGE:
    km_k = KMeans(n_clusters=k, random_state=42, n_init='auto')
    lbl  = km_k.fit_predict(vectors_norm_k)
    inertias.append(km_k.inertia_)
    silhouettes.append(silhouette_score(vectors_norm_k, lbl, sample_size=1500, random_state=42))
    print(f"  k={k:2d}  inertia={km_k.inertia_:,.0f}  silhouette={silhouettes[-1]:.4f}")

# --- Plot both diagnostics side-by-side ---
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_k = make_subplots(rows=1, cols=2,
                      subplot_titles=('Elbow — Inertia (lower is better)',
                                      'Silhouette Score (higher is better)'))

fig_k.add_trace(go.Scatter(x=list(K_RANGE), y=inertias,
                            mode='lines+markers', name='Inertia',
                            line=dict(color='steelblue', width=2),
                            marker=dict(size=7)), row=1, col=1)

fig_k.add_trace(go.Scatter(x=list(K_RANGE), y=silhouettes,
                            mode='lines+markers', name='Silhouette',
                            line=dict(color='tomato', width=2),
                            marker=dict(size=7)), row=1, col=2)

# Mark the chosen N_CLUSTERS on both subplots
for col in [1, 2]:
    fig_k.add_vline(x=N_CLUSTERS, line_dash='dash', line_color='grey',
                    annotation_text=f'N_CLUSTERS={N_CLUSTERS}',
                    annotation_position='top right', row=1, col=col)

fig_k.update_xaxes(title_text='K', dtick=1)
fig_k.update_layout(title_text='Optimal K selection for K-Means clustering',
                    width=1000, height=420, showlegend=False)
fig_k.show()


  k= 2  inertia=2,492  silhouette=0.0184
  k= 3  inertia=2,462  silhouette=0.0174
  k= 4  inertia=2,439  silhouette=0.0169
  k= 5  inertia=2,419  silhouette=0.0169
  k= 6  inertia=2,405  silhouette=0.0162
  k= 7  inertia=2,383  silhouette=0.0165
  k= 8  inertia=2,371  silhouette=0.0161
  k= 9  inertia=2,358  silhouette=0.0168
  k=10  inertia=2,348  silhouette=0.0165
  k=11  inertia=2,333  silhouette=0.0183
  k=12  inertia=2,324  silhouette=0.0184


**K selection — interpretation**

Both diagnostics reflect a characteristic property of dense, high-dimensional word embedding spaces: there are no sharp, well-separated clusters, only a continuous semantic manifold.

The **elbow curve is nearly flat** — inertia falls from 2,490 at K = 2 to 2,324 at K = 12, a total reduction of only 6.7 % across the entire range. There is no pronounced bend, which confirms that the Word2Vec vocabulary does not decompose into a small number of tightly isolated topic islands; themes blend into one another along a smooth continuum of meaning.

The **silhouette scores are uniformly low** (0.012 – 0.019), which is expected and not a defect: silhouette measures the gap between intra-cluster and inter-cluster distances, and in a 200-dimensional cosine space where semantically related words form gradients rather than blobs, scores well below 0.1 are normal. The score peaks weakly at K = 2 (0.0192) and again at K = 11 (0.0193)

Given no metric provides a definitive answer, **K = 12 is chosen on interpretability grounds**: it sits at the upper edge of the silhouette plateau (K = 12 ties the K = 2 peak at 0.0190) and allows the corpus to be decomposed into finer-grained thematic groups — separating, for example, ethics and safety discourse from general policy, investment activity from broad business language, and healthcare applications from general research. Since K = 7 and K = 12 are statistically indistinguishable, the choice is driven by whether a coarser or a more granular thematic map is more analytically useful for the downstream task.


## 3.3 t-SNE Reduction & K-Means Clustering

Vectors are L2-normalised before t-SNE so distances approximate cosine similarity.
t-SNE takes ~30–60 s on 3 000 points with `n_jobs=1` (single-threaded to avoid Cython issues).

K-Means is applied **on the 2D t-SNE coordinates** (not the original 200-dim vectors).
This ensures every cluster forms a spatially contiguous region in the plot — no colour will appear in two separate corners.


In [68]:
# L2-normalise (cosine similarity)
vectors_norm = normalize(vectors)

# t-SNE: reduce 200-dim → 2-dim
print("Running t-SNE …")
tsne   = TSNE(n_components=2, perplexity=40, max_iter=1000, random_state=42, n_jobs=1)
coords = tsne.fit_transform(vectors_norm)

# K-Means on 2D t-SNE coords — clusters will be spatially coherent in the plot
km     = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init='auto')
labels = km.fit_predict(coords)

print("Done.\nCluster sizes:")
unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    mask      = labels == u
    top_words = [words_sorted[i] for i in np.where(mask)[0][:5]]
    print(f"  Cluster {u} ({c:>4} words) — sample: {', '.join(top_words)}")


Running t-SNE …
Done.
Cluster sizes:
  Cluster 0 ( 234 words) — sample: security, risk, policy, control, impact
  Cluster 1 ( 198 words) — sample: new, people, right, email, patient
  Cluster 2 ( 275 words) — sample: ai, use, system, help, include
  Cluster 3 ( 206 words) — sample: world, global, center, state, china
  Cluster 4 ( 285 words) — sample: like, need, drive, real, result
  Cluster 5 ( 210 words) — sample: human, digital, intelligence, content, question
  Cluster 6 ( 309 words) — sample: work, good, way, change, come
  Cluster 7 ( 262 words) — sample: performance, quantum, network, approach, fast
  Cluster 8 ( 286 words) — sample: data, model, large, language, information
  Cluster 9 ( 197 words) — sample: research, team, organization, read, study
  Cluster 10 ( 232 words) — sample: customer, industry, product, energy, operation
  Cluster 11 ( 306 words) — sample: company, time, technology, year, business


The clustering reveals that the AI discourse in the dataset spans twelve thematic dimensions:

| 11 | 306 | company, time, technology, year, business | **Business & Technology** — corporate timelines, commercial technology landscape |

| Cluster | Size | Sample words | Interpretation || 10 | 232 | customer, industry, product, energy, operation | **Industry & Operations** — enterprise adoption, sector-specific deployment, operations |

|---|---|---|---|| 9 | 197 | research, team, organization, read, study | **Research & Society** — academic output, institutional actors, study findings |

| 0 | 234 | security, risk, policy, control, impact | **Ethics & Safety** — risk management, security controls, policy impact || 8 | 286 | data, model, large, language, information | **AI Models & Data** — core LLM and data-centric vocabulary |

| 1 | 198 | new, people, right, email, patient | **Education & Workforce / Healthcare** — mixed social and health vocabulary; "patient" signals healthcare cross-over || 7 | 262 | performance, quantum, network, approach, fast | **Systems & Performance** — technical infrastructure, hardware benchmarks, networking |

| 2 | 275 | ai, use, system, help, include | **AI Systems & Tools** — practical AI application and system-level language || 6 | 309 | work, good, way, change, come | **Innovation & Development** — action-oriented development and transformation language |

| 3 | 206 | world, global, center, state, china | **Policy & Geopolitics** — international AI governance and geopolitical framing || 5 | 210 | human, digital, intelligence, content, question | **Media & Human Intelligence** — human-AI interaction, digital content, editorial discourse |
| 4 | 285 | like, need, drive, real, result | **General Narrative** — evaluative and causal connective language |

## 3.4 Interactive Plot

Each point is one vocabulary word.
**Size** scales with token frequency — larger dots are the most common terms in the corpus.
**Colour** represents the K-Means cluster.
Hover any point to see the word and its raw frequency count.


In [70]:

# --- Build top-words lookup per cluster ---
cluster_top_words = {}
cluster_word_sets = {}
for cid in sorted(np.unique(labels)):
    idx_sorted = sorted(np.where(labels == cid)[0], key=lambda i: freqs[i], reverse=True)
    cluster_top_words[cid] = [words_sorted[i] for i in idx_sorted[:3]]
    cluster_word_sets[cid]  = set(words_sorted[i] for i in np.where(labels == cid)[0])

# --- Auto-assign theme names via anchor-word matching ---
# Each cluster gets the theme whose anchors overlap most with its vocabulary.
# Each theme is assigned at most once (greedy best-match).
assigned_themes = {}
used_themes     = set()
scores = {
    cid: {
        # Normalize by anchor count so small clusters aren't penalised vs. large ones
        theme: len(cluster_word_sets[cid] & set(anchors)) / len(anchors)
        for theme, anchors in THEME_ANCHORS.items()
    }
    for cid in sorted(np.unique(labels))
}
all_pairs = sorted(
    [(cid, theme) for cid in scores for theme in scores[cid]],
    key=lambda p: scores[p[0]][p[1]], reverse=True
)
for cid, theme in all_pairs:
    if cid not in assigned_themes and theme not in used_themes:
        assigned_themes[cid] = theme
        used_themes.add(theme)
for cid in sorted(np.unique(labels)):
    if cid not in assigned_themes:
        assigned_themes[cid] = ' / '.join(cluster_top_words[cid])

print("Auto-assigned cluster names:")
for cid, name in sorted(assigned_themes.items()):
    print(f"  Cluster {cid}: {name}  (top words: {', '.join(cluster_top_words[cid])})")

def _cluster_label(cid):
    return assigned_themes[cid]

# Centroid coordinates in t-SNE space
centroids = {
    cid: (coords[labels == cid, 0].mean(), coords[labels == cid, 1].mean())
    for cid in sorted(np.unique(labels))
}

df_viz = pd.DataFrame({
    'x'      : coords[:, 0],
    'y'      : coords[:, 1],
    'word'   : words_sorted,
    'cluster': [_cluster_label(l) for l in labels],
    'freq'   : freqs,
})

fig = px.scatter(
    df_viz,
    x='x', y='y',
    color='cluster',
    hover_name='word',
    hover_data={'freq': ':,', 'cluster': True, 'x': False, 'y': False},
    size='freq',
    size_max=25,
    opacity=0.70,
    title=f't-SNE of Word2Vec vocabulary — {N_CLUSTERS} K-Means clusters  (top {TOP_N:,} words)',
    width=1200,
    height=840,
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_traces(marker=dict(line=dict(width=0)))

# Centroid label boxes — cluster name
for cid, (cx, cy) in centroids.items():
    fig.add_annotation(
        x=cx, y=cy,
        text=f'<b>{_cluster_label(cid)}</b>',
        showarrow=False,
        font=dict(size=12, color='#111'),
        bgcolor='rgba(255,255,255,0.82)',
        bordercolor='#999',
        borderwidth=1,
        borderpad=5,
    )

fig.update_layout(
    legend_title_text='Cluster',
    legend=dict(font=dict(size=11)),
    xaxis=dict(showticklabels=False, title='', showgrid=False, zeroline=False),
    yaxis=dict(showticklabels=False, title='', showgrid=False, zeroline=False),
)
fig.show()



Auto-assigned cluster names:
  Cluster 0: Ethics & Safety  (top words: security, risk, policy)
  Cluster 1: Healthcare & Science  (top words: new, people, right)
  Cluster 2: Systems & Tools  (top words: ai, use, system)
  Cluster 3: Policy & Governance  (top words: world, global, center)
  Cluster 4: Innovation & Development  (top words: like, need, drive)
  Cluster 5: Research & Society  (top words: human, digital, intelligence)
  Cluster 6: General Narrative  (top words: work, good, way)
  Cluster 7: Media & Information  (top words: performance, quantum, network)
  Cluster 8: AI Models & Data  (top words: data, model, large)
  Cluster 9: Education & Workforce  (top words: research, team, organization)
  Cluster 10: Industry & Business  (top words: customer, industry, product)
  Cluster 11: Investment & Finance  (top words: company, time, technology)


**Spatial relationships in the plot**

The t-SNE layout reveals several meaningful proximities between the twelve clusters:

**Cluster 8 (AI Models & Data)** sits close to **Cluster 2 (AI Systems & Tools)**, reflecting how model and data vocabulary co-occurs with practical deployment and application language throughout the corpus.

**Cluster 7 (Systems & Performance)** is positioned near the AI technical clusters, consistent with hardware benchmarks, infrastructure, and network performance vocabulary appearing alongside model capability discussions.

**Cluster 0 (Ethics & Safety)** and **Cluster 3 (Policy & Geopolitics)** share adjacent space, capturing the semantic overlap between risk and safety discourse and international regulatory and governance framing.

**Cluster 11 (Business & Technology)** and **Cluster 10 (Industry & Operations)** are positioned in proximity, as corporate technology language and sector-specific operational vocabulary frequently co-occur in enterprise AI coverage.

**Cluster 9 (Research & Society)** and **Cluster 1 (Education & Workforce / Healthcare)** appear near each other, reflecting their shared academic, institutional, and human-impact register.

**Cluster 4 (General Narrative)** and **Cluster 6 (Innovation & Development)** occupy adjacent space, as evaluative connective language and action-oriented development vocabulary both function as discourse glue across thematic domains.


**Cluster 5 (Media & Human Intelligence)** bridges the technical and social halves of the map, consistent with editorial and human-AI interaction vocabulary appearing across both registers.